# Choosing feeds

A place is often served by several feeds, and many of them carry the same
buses and trains. This page shows how to list a place's feeds, compare them
side by side, and let `transitio` recommend which ones to fetch, for a place or
for an area of your own.

## A place's feeds

`feeds()` lists the feeds the index records for a place. Let's list
Augsburg's local and regional feeds:

In [1]:
import transitio

augsburg = transitio.place("Augsburg")
for feed in augsburg.feeds(tiers=["local", "regional"]):
    print(feed.feed_id, sorted(feed.tiers), feed.relevance_category, feed.service_start, feed.service_end)

f-mdb-3215 ['local', 'regional'] primary 2026-09-12 2026-12-12
f-germany~urban~transport ['local', 'regional'] primary 2026-10-03 2026-11-02
f-mdb-857 ['local'] primary 2024-12-15 2025-12-31
f-germany~regional~rail ['regional'] secondary 2026-10-03 2026-11-02
f-u281z9-mvv ['regional'] secondary 2026-09-01 2026-11-30
f-nvbw~ding ['regional'] secondary 2026-05-03 2026-12-12
f-nvbw~oam ['regional'] secondary 2026-05-03 2026-12-12
f-mdb-2393 ['regional'] secondary 2026-07-12 2026-12-12
f-nvbw~bodo ['regional'] secondary 2026-05-03 2026-12-12


Each feed also carries its relevance (`relevance_category` and `relevance`),
whether it has route shapes (`has_shapes`), its `license` and its
GTFS-realtime companions (`realtime`), the live-data feeds that go with a
static timetable.

Without `tiers`, `feeds()` returns the default view for the kind of place:

- a city or a metro: its primary and secondary feeds, that is, its local and
  regional service;
- a region: its secondary and tertiary feeds;
- a country: its tertiary feeds.

A region or country of at most 1,000 km², such as Monaco or San Juan, counts
as town-sized and keeps its primary, secondary and tertiary feeds.

You can also filter the list: `exclude` drops the named tiers, and
`requires=["shapes.txt"]` keeps only the feeds that carry those files.
`feed.realtime` lists the GTFS-realtime companions tied to a static feed,
and `Index.realtime_unlinked()` lists the companions the index could not tie
to one.

When `fetch(place=...)` without `tiers` finds the default view empty although
the place has feeds, `result.view_note` names those feeds and the `tiers`
that would fetch them, and a warning repeats the note.

To compare a place's feeds side by side, take them as a table. Tallinn is
served by several feeds:

In [2]:
tallinn = transitio.place("Tallinn")
table = tallinn.feeds(categories=None).to_dataframe()
table[["feed_id", "name", "covers", "repeats", "stop_count", "reason"]]

,feed_id,name,covers,repeats,stop_count,reason
0,f-ud-tallinn,TLT,0.908440,f-mdb-3235 56 %,18047,primary (local tier): 59 % of the departures s...
1,f-mdb-3235,Ühistranspordiregistri avaandmed,0.526374,"f-ud-tallinn 82 %, f-ud-harjumaa 13 %, f-ud-el...",18260,"primary (international, local, national, regio..."
2,f-ud-harjumaa,Harjumaa,0.087817,"f-mdb-3235 100 %, f-ud-kaugliinid 1 %",18254,"primary (local, regional tiers): 5 % of the de..."
3,f-ud-elron,ELRON,0.006950,f-mdb-3235 100 %,18054,"secondary (international, national, regional t..."
4,f-ud-raplamaa,Raplamaa,0.000311,f-mdb-3235 100 %,18247,secondary (regional tier): 0.019 % of the depa...
5,f-ud-laanemaa,Läänemaa,0.000062,f-mdb-3235 100 %,18247,secondary (regional tier): 0.0043 % of the dep...
6,f-ud-kaugliinid,Siseriiklikud kaugbussiliinid,0.001581,"f-mdb-3235 100 %, f-ud-harjumaa 8 %",18071,"secondary (national, regional tiers): 0.12 % o..."
7,f-ud-laanevirumaa,Lääne-Virumaa,0.000105,f-mdb-3235 100 %,18255,secondary (regional tier): 0.0063 % of the dep...
8,f-ud-parnumaa,Pärnumaa,0.000088,f-mdb-3235 100 %,18068,secondary (regional tier): 0.0064 % of the dep...


The table has one row per feed, with each feed's tiers, relevance, service,
modes, timetable window, access and stop count, among other columns. These
columns help most when you compare feeds:

- `share_of_place` is the feed's share of the departures summed over the
  place's feeds, or its share of the stops where the index measured stops.
- `covers` is the share of the place's departures the feed runs, each
  departure counted once.
- `repeats` names the other feeds that also run the feed's departures.
- `stale_when_indexed` gives the day the feed's timetable ended, when that was
  more than 30 days before the index crawled the feed.
- `contained_in` names the larger feeds whose stops and routes contain this
  feed's.
- `reason` sums up the row in words.

`covers` and `repeats` need an index that records which feeds run the same
lines; without that evidence, both are empty.

## Which feeds to use

Several feeds often run the same service: a city's own feed, a national
aggregate and a regional one may each carry nearly all of a city's
departures. `recommend()` says which feeds to take for a day and why it
leaves out the others:

In [3]:
rec = tallinn.recommend("2026-10-14")
print(rec)

Tallinn (city), 2026-10-14: take 2 feeds, covering about 100 % of the departures the index records there, each counted once
  + f-ud-tallinn (TLT): covers 90 % of the place's departures (91 % of bus; 79 % of rail, subway and tram)
  + f-mdb-3235 (Ühistranspordiregistri avaandmed): adds bus the feeds above lack: 8 % of the place's bus
  - f-ud-harjumaa, f-ud-elron, f-ud-kaugliinid, f-ud-raplamaa, f-ud-parnumaa, f-ud-laanevirumaa, f-ud-laanemaa: repeats f-mdb-3235 (100 % of its departures)


As we can see, `recommend()` takes two feeds for Tallinn and leaves out the
others, because the taken feeds already run their departures. Every feed
serving the place is a candidate, whatever its category. `recommend()` then
chooses in three steps:

1. **It leaves out the feeds that cannot serve the day.** These are the feeds
   that were stale when indexed, those whose timetable as indexed does not
   run on the day (today when no day is given), and those that need a paid
   account.
2. **It takes feeds one at a time.** Counting each of the place's departures
   once, it takes the feed that adds the most departures the taken feeds
   lack. It stops when the taken feeds cover 95 % of the departures
   (`target=`) and 80 % of each mode, with rail, subway and tram counting as
   one mode. It takes at most four feeds (`max_feeds=`).
3. **It breaks ties.** Among feeds that add about as much, the open one with
   the fewest stops wins.

Steps 2 and 3 need an index that records which feeds run the same lines.
Without that evidence, `recommend()` takes the feed with the most departures
and marks the others "not compared".

The recommendation's `feed_ids` lists the taken feeds, and `to_dataframe()`
gives every feed with the reason it was taken or left out:

In [4]:
print(rec.feed_ids)
rec.to_dataframe()

['f-ud-tallinn', 'f-mdb-3235']


,feed_id,name,taken,adds,reason
0,f-ud-tallinn,TLT,True,0.90844,covers 90 % of the place's departures (91 % of...
1,f-mdb-3235,Ühistranspordiregistri avaandmed,True,0.09156,adds bus the feeds above lack: 8 % of the plac...
2,f-ud-harjumaa,Harjumaa,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
3,f-ud-elron,ELRON,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
4,f-ud-kaugliinid,Siseriiklikud kaugbussiliinid,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
5,f-ud-raplamaa,Raplamaa,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
6,f-ud-parnumaa,Pärnumaa,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
7,f-ud-laanevirumaa,Lääne-Virumaa,False,0.00000,repeats f-mdb-3235 (100 % of its departures)
8,f-ud-laanemaa,Läänemaa,False,0.00000,repeats f-mdb-3235 (100 % of its departures)


To fetch only the feeds it takes, pass the recommendation to `fetch`,
which takes the place and the day from it:

```python
result = transitio.fetch(feeds=rec)
```

[Fetching data](fetching_data.ipynb) shows this call and its options.

## Feeds for an area

`transitio.index.area` finds the index places that cover an area of your
own, such as a study region drawn as a polygon. Let's combine Munich and the
district around it into one study region:

In [5]:
import shapely

munich = transitio.place("Munich", kind="city")
district = transitio.place("Munich District")
region = shapely.union_all([munich.geometry, district.geometry])
area = transitio.index.area(region)
area.coverage          # the share of the area's land the parts cover

1.0

`area.parts` lists the places the area is made of:

In [6]:
for part in area.parts:
    print(part.place, part.whole, round(part.inside, 2), round(part.holds, 2))

Place(tp_928, region, 'Munich District') True 1.0 0.68
Place(tp_37, city, 'Munich') True 1.0 0.32


Notice that Munich and its district are the two parts of the area, while
Bavaria, which contains both, is not a part. `area` chooses the parts by
these rules:

- Only cities, regions and countries with feeds count; metros do not.
- A place at least half inside the area is a **whole part**, unless a place
  containing it is also at least half inside.
- A place less than half inside the area is a **partial part** when it holds
  at least 1 % of the area and no place containing it is a whole part.
- A partial part gives way to the smaller places inside it that are parts.
  This is why Bavaria is not a part here, and why a box inside one district
  gives that district, not its city or region.

In the loop above, `inside` is the share of the place inside the area, and
`holds` is the share of the area inside the place. `coverage` divides the
area the parts cover by the area's land, that is, its part inside the
index's countries: sea does not count, and land in a country without feeds
counts as uncovered. `country="DE"` keeps only the places of one country.

`area.feeds()` asks each part for its feeds, as `place.feeds()` does and
with the same arguments, and lists each feed once:

In [7]:
[feed.feed_id for feed in area.feeds()]

['f-mdb-2333',
 'f-u281z9-mvv',
 'f-mdb-3215',
 'f-germany~urban~transport',
 'f-mdb-779',
 'f-germany~regional~rail',
 'f-nvbw~ding',
 'f-nvbw~bodo',
 'f-nvbw~oam',
 'f-mdb-2393']

A box inside a city gives the districts the index holds for it, each a place
of its own; a district's `parent` names the city it lies in. Let's take the
centre of Turku:

In [8]:
centre = transitio.index.area((22.20, 60.42, 22.32, 60.48))
print(len(centre.parts), "parts:", [part.place.name for part in centre.parts[:5]])
print(centre.parts[0].place.parent)

40 parts: ['Pitkämäki', 'Nummi', 'Satama', 'Vasaramäki', 'Lauttaranta']
Place(tp_17305, city, 'Turku')


As we can see, the box over the centre of Turku gives 40 districts, each a
place of its own inside Turku.

`area.recommend()` works like `place.recommend()`, summing each feed's
departures over the parts of the area.

`fetch` uses the same parts when you give it an area:

- When the parts cover at least half of the area's land, `fetch` takes their
  feeds, with the same `tiers`, `exclude` and `on_unknown` as for a place,
  and `result.places` lists the parts.
- Otherwise, it searches the Mobility Database catalogue by bounding box and
  warns, for example `"the feed index's places cover 20% of the area; 3 feeds
  from the Mobility Database catalogue by bounding box"`.
- With `index=False`, it searches the catalogue without the index and without
  the warning.

The OpenStreetMap extract covers the area itself.

```python
result = transitio.fetch(region, tiers=["local", "regional"], osm=False)
[place.name for place in result.places]   # ['Munich District', 'Munich']
```